In [1]:
from transport.routes_generator import citygraph_dataset
from transport.routes_generator import inductive_route_learning, eval_route_generator, bee_colony
from omegaconf import OmegaConf, DictConfig
from tqdm import tqdm
from pathlib import Path
from transport.routes_generator.utils import get_eval_cfg

# Read data

## Read Mandl

In [ ]:
from pathlib import Path
import torch
import numpy as np

cwd = Path.cwd()
print("Current directory:", cwd)

# Путь к данным (две папки вверх до корня репозитория, потом в transport/data)
data_dir = cwd.parent.parent / "examples/data//banchmark"

# Instance name
instance_name = "Mandl"

# --- Reading coordinates ---
coords_path = data_dir / f"{instance_name}Coords.txt"
node_locs_np = np.genfromtxt(coords_path, skip_header=1)
node_locs = torch.tensor(node_locs_np, dtype=torch.float32)
orig_pos = node_locs.clone()  # original positions

# --- Reading travel times ---
tt_path = data_dir / f"{instance_name}TravelTimes.txt"
street_adj_np = np.genfromtxt(tt_path)
street_adj = torch.tensor(street_adj_np, dtype=torch.float32) * 60  # convert to seconds

# --- Reading demand ---
dmd_path = data_dir / f"{instance_name}Demand.txt"
demand_np = np.genfromtxt(dmd_path)
demand = torch.tensor(demand_np, dtype=torch.float32)

# --- Printing tensor shapes ---
print("Coords tensor shape:", node_locs.shape)
print("Travel times tensor shape:", street_adj.shape)
print("Demand tensor shape:", demand.shape)

# --- Preparing input dictionary for the model ---
input_tensors = {
    'street_adj': street_adj,
    'demand': demand,
    'node_locs': node_locs
}


Current directory: /root/transport/examples/route_generator
Coords tensor shape: torch.Size([15, 2])
Travel times tensor shape: torch.Size([15, 15])
Demand tensor shape: torch.Size([15, 15])


## Read Tartu

In [4]:
import pickle
import pandas as pd
import numpy as np
import torch
import networkx as nx

from pathlib import Path
import pickle
import pandas as pd
import torch
import numpy as np
import networkx as nx

# Текущая рабочая директория
cwd = Path.cwd()
print("Current directory:", cwd)

# Путь к данным
data_dir = cwd.parent.parent / "examples/data/tartu/"

# --- 1. Загрузка графа G ---
file_path = data_dir / "bus_graph_Tartu.pkl"
with open(file_path, "rb") as f:
    G = pickle.load(f)

# --- 2. Загрузка OD-матрицы ---
od_path = data_dir / "OD_matrix_Tartu.csv"
OD = pd.read_csv(od_path).set_index("Unnamed: 0", drop=True)
demand_np = OD.values
demand = torch.tensor(demand_np, dtype=torch.float32)

# --- 3. Получение списка узлов в правильном порядке ---
nodes = sorted(G.nodes())  # Важно: сортировка, чтобы индексы совпадали с OD
n_nodes = len(nodes)
node_to_idx = {node: i for i, node in enumerate(nodes)}

# --- 4. Матрица координат (node_locs): [x, y] ---
node_locs_list = []
for node in nodes:
    x = G.nodes[node]['x']
    y = G.nodes[node]['y']
    node_locs_list.append([x, y])

node_locs_np = np.array(node_locs_list, dtype=np.float32)
node_locs = torch.tensor(node_locs_np)

# --- 5. Симметричная матрица весов по time_min (street_adj) ---
inf = float('inf')
adj_matrix = np.full((n_nodes, n_nodes), inf, dtype=np.float32)
np.fill_diagonal(adj_matrix, 0.0)  # диагональ = 0

for u, v, data in G.edges(data=True):
    if 'time_min' in data:
        time = data['time_min']
        i = node_to_idx[u]
        j = node_to_idx[v]
        adj_matrix[i, j] = time
        adj_matrix[j, i] = time  # симметричность

street_adj = torch.tensor(adj_matrix)
# Замена np.inf → torch.inf
street_adj[street_adj == inf] = torch.inf

# --- 6. Вывод результата ---
input_tensors = {
    'street_adj': street_adj,
    'demand': demand,
    'node_locs': node_locs
}

print("Готово! Формат:")
print(f"street_adj: {street_adj.shape} → {street_adj.dtype}")
print(f"demand: {demand.shape} → {demand.dtype}")
print(f"node_locs: {node_locs.shape} → {node_locs.dtype}")


Current directory: /root/transport/examples/route_generator
Готово! Формат:
street_adj: torch.Size([199, 199]) → torch.float32
demand: torch.Size([199, 199]) → torch.float32
node_locs: torch.Size([199, 2]) → torch.float32


# Run LC

In [ ]:
from omegaconf import OmegaConf
from pathlib import Path

# Parameters (example values)
params = {
    "dataset_name": "tensor",
    "n_routes": 6,
    "min_route_len": 2,
    "max_route_len": 8,
    "demand_time_weight": 0.33,
    "route_time_weight": 0.33,
    "median_connectivity_weight": 0.33,
    "run_name": "test_run",
}

# Path to config directory relative to current notebook
cwd = Path.cwd()
cfg_dir = cwd.parent.parent / "transport/routes_generator/cfg"
cfg_dir = str(cfg_dir)

# --- Path to model weights ---
model_weights_path = cwd.parent.parent / "examples/data/model_weights/inductive_random_graphs_weighted_connectivity.pt"
params["model_weights"] = str(model_weights_path)


# Load evaluation config (assuming get_eval_cfg is defined elsewhere)
cfg_eval = get_eval_cfg(
    cfg_dir=cfg_dir,
    base_cfg_name='eval_model_mumford',
    params=params
)

# Print config in YAML format
print(OmegaConf.to_yaml(cfg_eval))


experiment:
  logdir: null
  anomaly: false
  cpu: false
  seed: 0
  symmetric_routes: true
  cost_function:
    type: mine
    kwargs:
      mean_stop_time_s: 0
      avg_transfer_wait_time_s: 300
      demand_time_weight: 0.33
      route_time_weight: 0.33
      median_connectivity_weight: 0.33
      constraint_violation_weight: 5.0
      use_weighted_connectivity: true
      variable_weights: true
      pp_fraction: 0.15
      op_fraction: 0.15
      mcw_fraction: 0.15
model:
  common:
    dropout: 0.0
    nonlin_type: ReLU
    embed_dim: 64
  route_generator:
    kwargs:
      force_linking_unlinked: false
      logit_clip: null
      n_nodepair_layers: 3
      n_pathscorer_layers: 3
      pathscorer_hidden_dim: 16
      n_halt_layers: 3
      halt_scorer_type: endpoints
      serial_halting: true
    type: PathCombiningRouteGenerator
  backbone_gn:
    net_type: graph attn
    kwargs:
      n_layers: 5
      in_node_dim: 4
      in_edge_dim: 14
      use_norm: false
      n_heads:

In [13]:
from torch_geometric.loader import DataLoader
from transport.routes_generator.citygraph_dataset import get_dataset_from_config
import transport.routes_generator.utils as lrnu
from transport.routes_generator.eval_route_generator import eval_model
from transport.routes_generator.torch_utils import dump_routes

cfg = cfg_eval

# Если у тебя есть заранее подготовленные тензоры — можешь загрузить здесь:
tensors = input_tensors

# Стандартная обработка конфига и модели
assert 'model' in cfg, "Must provide config for model!"
DEVICE, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
    cfg, 'nn_construction_', weights_required=True
)

# Загружаем тестовый датасет
test_ds = get_dataset_from_config(cfg.eval.dataset, tensors=tensors)
test_dl = DataLoader(test_ds, batch_size=cfg.batch_size)

# Запускаем оценку модели
n_samples = cfg.get('n_samples', None)
sbs = cfg.get('sample_batch_size', cfg.batch_size)


_, unserved_demand, metrics, routes = eval_model(
    model,
    test_dl,
    cfg.eval,
    cost_obj,
    n_samples=n_samples,
    sample_batch_size=sbs,
    return_routes=True,
    silent=True,
    device=DEVICE
)

# Сохраняем маршруты
dump_routes(run_name, routes.cpu())

# Печатаем результат в консоль
print("=== Evaluation done ===")
print(f"Run name: {run_name}")
print(f"Unserved demand: {unserved_demand}")
print(f"Metrics: {metrics}")
print(routes)

=== Evaluation done ===
Run name: nn_construction_test_run
Unserved demand: tensor([[[0.0000e+00, 8.7426e+00, 1.2635e-01,  ..., 5.6898e+00,
          1.1788e-01, 2.7935e+00],
         [8.0729e+00, 0.0000e+00, 1.7791e+00,  ..., 1.3244e+02,
          1.6295e+00, 6.0707e+01],
         [1.1458e-01, 1.7471e+00, 0.0000e+00,  ..., 1.9662e+00,
          3.8255e-01, 1.0190e+00],
         ...,
         [2.1393e+00, 5.3928e+01, 8.1524e-01,  ..., 0.0000e+00,
          7.4514e-01, 6.5929e+01],
         [2.3921e-01, 3.5809e+00, 8.5605e-01,  ..., 4.0217e+00,
          0.0000e+00, 2.1025e+00],
         [7.7934e+00, 1.8341e+02, 3.1349e+00,  ..., 4.8918e+02,
          2.8904e+00, 0.0000e+00]]])
Metrics: {'cost': tensor([6.3728]), 'ATT': tensor([0.9547]), 'RTT': tensor([2.4379]), '$d_0$': tensor([1.4691]), '$d_1$': tensor([0.3155]), '$d_2$': tensor([0.]), '$d_{un}$': tensor([98.2154]), '# disconnected node pairs': tensor([18416.]), '# stops out of bounds': tensor([0.]), 'median_connectivity': tensor([0.0

/root/transport/transport/routes_generator/utils.py:319: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


# Run NBCO 

In [14]:
from omegaconf import OmegaConf
import os

# params полностью повторяет значения из твоего примера
params = {
    "dataset_name": "tensor",
    "n_routes": 6,
    "min_route_len": 2,
    "max_route_len": 8,
    "demand_time_weight": 0.33,
    "route_time_weight": 0.33,
    "median_connectivity_weight": 0.33,
    "run_name": "test_run",
    "starting_routes_init":"tensor"
}

# Path to config directory relative to current notebook
cwd = Path.cwd()
cfg_dir = cwd.parent.parent / "transport/routes_generator/cfg"
cfg_dir = str(cfg_dir)

# --- Path to model weights ---
# Suppose model is in transport/routes_generator/
model_weights_path = cwd.parent.parent / "transport/routes_generator/inductive_random_graphs_weighted_connectivity.pt"
params["model_weights"] = str(model_weights_path)

cfg_bee = get_eval_cfg(
    cfg_dir=cfg_dir,
    base_cfg_name='neural_bco_mumford',
    params=params
)

# print(OmegaConf.to_yaml(cfg_bee))


In [15]:
from torch_geometric.loader import DataLoader
from transport.routes_generator.citygraph_dataset import get_dataset_from_config
import transport.routes_generator.utils as lrnu

use_neural_bees = cfg_bee.get('neural_bees', False)
if use_neural_bees:
    prefix = 'neural_bco_'
else:
    prefix = 'bco_'

DEVICE, run_name, sum_writer, cost_obj, bee_model = \
    lrnu.process_standard_experiment_cfg(cfg_bee, prefix, 
                                            weights_required=True)

# read in the dataset
test_ds = get_dataset_from_config(cfg_bee.eval.dataset, tensors=input_tensors)
test_dl = DataLoader(test_ds, batch_size=cfg_bee.batch_size)

force_linking_unlinked = cfg_bee.get('force_linking_unlinked', False)

if not use_neural_bees:
    bee_model = None
elif bee_model is not None:
    bee_model.force_linking_unlinked = force_linking_unlinked
    bee_model.eval()

nt1b = cfg_bee.get('n_type1_bees', None)
nt2b = cfg_bee.get('n_type2_bees', None)
test_output = \
    lrnu.test_method(bee_colony, test_dl, cfg_bee.eval, cfg_bee.init, cost_obj, 
        sum_writer=sum_writer, silent=True, n_bees=cfg_bee.n_bees,
        n_iterations=cfg_bee.n_iterations, n_type1_bees=nt1b, n_type2_bees=nt2b,  
        device=DEVICE, bee_model=bee_model, return_routes=True,
        force_linking_unlinked=force_linking_unlinked, routes_tensor=routes)
routes = test_output[-1]
metrics = test_output[-2]
unserved_demand = test_output[-3]

# Печатаем результат в консоль
print("=== Evaluation done ===")
print(f"Run name: {run_name}")
print(f"Unserved demand: {unserved_demand}")
print(f"Metrics: {metrics}")
print(routes)

=== Evaluation done ===
Run name: neural_bco_test_run
Unserved demand: tensor([[[0.0000e+00, 8.7426e+00, 1.2635e-01,  ..., 5.6898e+00,
          1.1788e-01, 2.7935e+00],
         [8.0729e+00, 0.0000e+00, 1.7791e+00,  ..., 1.3244e+02,
          1.6295e+00, 6.0707e+01],
         [1.1458e-01, 1.7471e+00, 0.0000e+00,  ..., 1.9662e+00,
          3.8255e-01, 1.0190e+00],
         ...,
         [2.1393e+00, 5.3928e+01, 8.1524e-01,  ..., 0.0000e+00,
          7.4514e-01, 6.5929e+01],
         [2.3921e-01, 3.5809e+00, 8.5605e-01,  ..., 4.0217e+00,
          0.0000e+00, 2.1025e+00],
         [7.7934e+00, 1.8341e+02, 3.1349e+00,  ..., 4.8918e+02,
          2.8904e+00, 0.0000e+00]]])
Metrics: {'cost': tensor([6.2627]), 'ATT': tensor([0.0523]), 'RTT': tensor([1.1201]), '$d_0$': tensor([0.9876]), '$d_1$': tensor([0.]), '$d_2$': tensor([0.]), '$d_{un}$': tensor([99.0124]), '# disconnected node pairs': tensor([18551.5000]), '# stops out of bounds': tensor([0.]), 'median_connectivity': tensor([0.0533])